# Module 04: Attitude Dynamics & Kinematics

Attitude (orientation) is one of the most important aspects of spacecraft simulation. This module covers the math and Basilisk utilities for describing, converting, and propagating spacecraft attitude.

### Learning Objectives
- Understand attitude representations: DCM, Euler angles, quaternions, MRP
- Use `RigidBodyKinematics` for all conversions
- Simulate Euler's rotational equations of motion
- Understand Basilisk's automatic MRP shadow set switching
- Visualize attitude trajectories in 3D

---

## 1. Attitude Representations

| Representation | Params | Pros | Cons |
|---|---|---|---|
| Direction Cosine Matrix (DCM) | 9 (3×3) | Intuitive, direct rotation | Redundant, large |
| Euler Angles (e.g. 3-2-1) | 3 | Intuitive for small angles | Gimbal lock singularity |
| Quaternion | 4 | Singularity-free, compact | Non-intuitive, unit constraint |
| MRP (σ) | 3 | Compact, smooth, BSK native | Singularity at |σ|=1 → shadow set |

Basilisk uses **MRP** internally. All conversions are in `RigidBodyKinematics`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('plots', exist_ok=True)

from Basilisk.utilities import RigidBodyKinematics as rbk
from Basilisk.utilities import macros

print("RigidBodyKinematics imported.")

# Key functions available in rbk:
funcs = [f for f in dir(rbk) if not f.startswith('_')]
print(f"\n{len(funcs)} functions available in RigidBodyKinematics:")
print(", ".join(funcs[:20]), "...")

---

## 2. Direction Cosine Matrix (DCM)

The DCM [BN] rotates vectors from frame N to frame B: **v_B = [BN] · v_N**

In [ ]:
# ── Rotation about Z-axis by 30 degrees ───────────────────────────────────────
angle_deg = 30.0
angle_rad = angle_deg * macros.D2R

# Using rbk elementary rotation
BN = rbk.Mi(angle_rad, 3)   # M3 = rotation about 3rd axis (Z)
print("DCM for 30° rotation about Z:")
print(np.round(BN, 4))

# Verify: rotate X-axis unit vector
v_N = np.array([1.0, 0.0, 0.0])
v_B = BN @ v_N
print(f"\nX-axis in N: {v_N}")
print(f"X-axis in B: {np.round(v_B, 4)}")
print(f"Expected:    [{np.cos(angle_rad):.4f}, {-np.sin(angle_rad):.4f}, 0.0]")

---

## 3. MRP Conversions

In [ ]:
# ── DCM <-> MRP ───────────────────────────────────────────────────────────────
# Convert a DCM (from Z-rotation above) to MRP
sigma = rbk.C2MRP(BN)
print(f"MRP from DCM (30° Z-rot): {sigma}")

# Convert back to DCM
BN_recovered = rbk.MRP2C(sigma)
print(f"\nDCM recovered from MRP:")
print(np.round(BN_recovered, 6))
print(f"Max error vs original: {np.max(np.abs(BN_recovered - BN)):.2e}")

# ── Quaternion <-> MRP ────────────────────────────────────────────────────────
q = rbk.MRP2EP(sigma)   # EP = Euler Parameters = quaternion [q1, q2, q3, q4]
print(f"\nQuaternion (q1,q2,q3,q4): {np.round(q, 6)}")
print(f"|q| = {np.linalg.norm(q):.8f}  (should be 1.0)")

sigma_back = rbk.EP2MRP(q)
print(f"MRP recovered from quaternion: {np.round(sigma_back, 6)}")

# ── Euler angles (3-2-1 ZYX) <-> MRP ─────────────────────────────────────────
# Tait-Bryan angles: yaw (ψ), pitch (θ), roll (φ)
yaw_deg, pitch_deg, roll_deg = 30.0, 15.0, 10.0
euler_321 = np.array([yaw_deg, pitch_deg, roll_deg]) * macros.D2R

C_321 = rbk.euler3212C(euler_321)   # 3-2-1 Euler to DCM
sigma_321 = rbk.C2MRP(C_321)
print(f"\n3-2-1 Euler [{yaw_deg}, {pitch_deg}, {roll_deg}]° -> MRP: {np.round(sigma_321, 6)}")

---

## 4. MRP Shadow Set

When |**σ**| → 1, MRP approaches a singularity. Basilisk automatically switches to the *shadow set*:
```
σ_s = -σ / |σ|²
```
The shadow set represents the same physical rotation.

In [ ]:
# ── MRP shadow set demo ───────────────────────────────────────────────────────
# Large rotation: 350° about Z (|sigma| will be close to 1)
angle_large = 350.0 * macros.D2R
C_large = rbk.Mi(angle_large, 3)
sigma_large = rbk.C2MRP(C_large)
print(f"MRP for 350° Z-rotation: {np.round(sigma_large, 4)}")
print(f"|sigma| = {np.linalg.norm(sigma_large):.6f}")

# Shadow set: MRPswitch(sigma, s2) returns the shadow set when |sigma| > s2
# Using s2=1.0 mirrors Basilisk's automatic switching threshold
sigma_s = rbk.MRPswitch(sigma_large, 1.0)
print(f"Shadow MRP:              {np.round(sigma_s, 4)}")
print(f"|sigma_s| = {np.linalg.norm(sigma_s):.6f}")

# Verify: both give the same DCM
C_from_sigma   = rbk.MRP2C(sigma_large)
C_from_sigma_s = rbk.MRP2C(sigma_s)
err = np.max(np.abs(C_from_sigma - C_from_sigma_s))
print(f"\nDCM difference between sigma and shadow: {err:.2e}  (should be ~0)")

---

## 5. Euler's Equations of Rotational Motion

For a rigid body in torque-free motion, **Euler's equations** describe angular velocity evolution:

$$
[I]\dot{\boldsymbol{\omega}} = -[\boldsymbol{\omega}\times][I]\boldsymbol{\omega}
$$

And the **MRP kinematic equation** propagates attitude from angular velocity:

$$
\dot{\boldsymbol{\sigma}} = \frac{1}{4}\left[(1-|\boldsymbol{\sigma}|^2)[I_3] + 2[\boldsymbol{\sigma}\times] + 2\boldsymbol{\sigma}\boldsymbol{\sigma}^T\right]\boldsymbol{\omega}
$$

Below we integrate these numerically to simulate a tumbling spacecraft (no Basilisk needed for this).

In [ ]:
from scipy.integrate import solve_ivp

# ── Inertia tensor (asymmetric body) ─────────────────────────────────────────
I_body = np.diag([100.0, 400.0, 300.0])   # kg*m^2
I_inv  = np.linalg.inv(I_body)

def skew(v):
    """Skew-symmetric (cross-product) matrix of vector v."""
    return np.array([[0, -v[2], v[1]],
                     [v[2], 0, -v[0]],
                     [-v[1], v[0], 0]])

def mrp_rate(sigma, omega):
    """MRP kinematic equation: sigma_dot = B(sigma) * omega / 4."""
    s2 = sigma @ sigma
    B  = (1 - s2) * np.eye(3) + 2 * skew(sigma) + 2 * np.outer(sigma, sigma)
    return 0.25 * B @ omega

def equations_of_motion(t, state):
    sigma = state[0:3]
    omega = state[3:6]

    # Switch to shadow set if |sigma| > 1 (avoid singularity)
    # MRPswitch(sigma, s2) returns shadow when |sigma| > s2
    if np.linalg.norm(sigma) > 1.0:
        sigma = rbk.MRPswitch(sigma, 1.0)

    omega_dot = I_inv @ (-skew(omega) @ I_body @ omega)
    sigma_dot = mrp_rate(sigma, omega)

    return np.concatenate([sigma_dot, omega_dot])

# ── Initial conditions ────────────────────────────────────────────────────────
sigma0 = np.array([0.1, 0.2, -0.1])
omega0 = np.array([0.05, 0.03, 0.20])  # rad/s
state0 = np.concatenate([sigma0, omega0])

# ── Integrate ─────────────────────────────────────────────────────────────────
t_span = (0, 300)   # 300 seconds
t_eval = np.linspace(*t_span, 3000)
sol    = solve_ivp(equations_of_motion, t_span, state0, t_eval=t_eval,
                   method='RK45', rtol=1e-9, atol=1e-11)

t       = sol.t
sigma   = sol.y[0:3].T
omega   = sol.y[3:6].T
sigma_n = np.linalg.norm(sigma, axis=1)

print(f"Integration complete. {len(t)} points.")
print(f"|sigma| range: {sigma_n.min():.4f} — {sigma_n.max():.4f}")

In [ ]:
# ── Plot angular velocity (Euler's equations) ─────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
ax_labels = [r'$\omega_x$ (deg/s)', r'$\omega_y$ (deg/s)', r'$\omega_z$ (deg/s)']
colors = ['tab:blue', 'tab:orange', 'tab:green']

for i in range(3):
    axes[i].plot(t, np.degrees(omega[:, i]), color=colors[i])
    axes[i].set_ylabel(ax_labels[i])
    axes[i].grid(True, alpha=0.3)

axes[0].set_title('Angular Velocity — Torque-Free Tumble (Euler Equations)')
axes[2].set_xlabel('Time (s)')
plt.tight_layout()
plt.savefig('plots/04_euler_omega.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Plot MRP attitude ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for i in range(3):
    axes[0].plot(t, sigma[:, i], color=colors[i], label=[r'$\sigma_1$', r'$\sigma_2$', r'$\sigma_3$'][i])
axes[0].set_ylabel('MRP components')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_title('MRP Attitude — Torque-Free Tumble')

axes[1].plot(t, sigma_n, color='black')
axes[1].axhline(1.0, color='red', linestyle='--', label='Shadow set threshold (|σ|=1)')
axes[1].set_ylabel('|σ| (MRP magnitude)')
axes[1].set_xlabel('Time (s)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/04_mrp_tumble.png', dpi=100, bbox_inches='tight')
plt.show()

---

## 6. Verify Rotational Kinetic Energy Conservation

For torque-free motion, the rotational kinetic energy **T** and angular momentum magnitude **|H|** must both be constant.

In [ ]:
T = np.array([0.5 * omega[k] @ I_body @ omega[k] for k in range(len(t))])
H = np.array([I_body @ omega[k] for k in range(len(t))])
H_norm = np.linalg.norm(H, axis=1)

T_err = (T - T[0]) / T[0] * 100
H_err = (H_norm - H_norm[0]) / H_norm[0] * 100

print(f"Kinetic energy conservation error: {abs(T_err).max():.2e} %")
print(f"Angular momentum conservation error: {abs(H_err).max():.2e} %")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(t, T_err, color='purple')
axes[0].axhline(0, color='gray', linestyle='--')
axes[0].set_title('Rotational KE Conservation Error (%)')
axes[0].set_xlabel('Time (s)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, H_err, color='teal')
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].set_title('Angular Momentum Conservation Error (%)')
axes[1].set_xlabel('Time (s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/04_conservation.png', dpi=100, bbox_inches='tight')
plt.show()

---

## 7. Reference Frame Relative Attitude

In practice, we care about the attitude **relative to a reference frame** (e.g., LVLH, Sun-pointing). Given:
- **σ_BN**: body-to-inertial MRP
- **σ_RN**: reference-to-inertial MRP

The error MRP is: **σ_BR** (body relative to reference)

In [ ]:
# ── Compute attitude error relative to a nadir-pointing reference ─────────────
sigma_BN = np.array([0.1, -0.2, 0.3])   # current body attitude
sigma_RN = np.array([0.05, -0.15, 0.28]) # desired reference attitude

# C_BR = C_BN * C_NR = C_BN * C_RN^T
C_BN = rbk.MRP2C(sigma_BN)
C_RN = rbk.MRP2C(sigma_RN)
C_BR = C_BN @ C_RN.T

sigma_BR = rbk.C2MRP(C_BR)

# Or equivalently using subMRP:
sigma_BR_alt = rbk.subMRP(sigma_BN, sigma_RN)

print(f"Body MRP sigma_BN:        {np.round(sigma_BN, 4)}")
print(f"Reference MRP sigma_RN:   {np.round(sigma_RN, 4)}")
print(f"Error MRP sigma_BR:       {np.round(sigma_BR, 4)}")
print(f"Error MRP (via subMRP):   {np.round(sigma_BR_alt, 4)}")
print(f"\nAttitude error magnitude: {np.degrees(4*np.arctan(np.linalg.norm(sigma_BR))):.3f} deg")

---

## `RigidBodyKinematics` Quick Reference

| Function | Description |
|---|---|
| `rbk.Mi(angle, axis)` | Elementary rotation DCM (axis = 1,2,3) |
| `rbk.MRP2C(sigma)` | MRP → DCM |
| `rbk.C2MRP(C)` | DCM → MRP |
| `rbk.MRP2EP(sigma)` | MRP → quaternion [q1,q2,q3,q4] |
| `rbk.EP2MRP(q)` | Quaternion → MRP |
| `rbk.euler3212C(angles)` | 3-2-1 Euler → DCM |
| `rbk.C2Euler321(C)` | DCM → 3-2-1 Euler |
| `rbk.MRPswitch(sigma, s2)` | Return shadow set when \|σ\| > s2 (use s2=1.0) |
| `rbk.subMRP(s1, s2)` | Relative MRP: s1 relative to s2 |
| `rbk.addMRP(s1, s2)` | Compound two MRP rotations |
| `rbk.BmatMRP(sigma)` | Kinematic B matrix: σ̇ = B(σ)ω/4 |

---

## Summary

- Basilisk uses **MRP** as its native attitude parameterization
- `RigidBodyKinematics` handles all conversions (DCM, quaternion, Euler angles, MRP)
- The shadow set avoids the MRP singularity at |σ|=1 — use `rbk.MRPswitch(sigma, 1.0)`; Basilisk handles this automatically internally
- Attitude error = `subMRP(sigma_BN, sigma_RN)` — this feeds the control error in FSW modules

**Next: [05 - Flight Software & Attitude Control](05_fsw_attitude_control.ipynb)**